!nvidia-smi: Runs the NVIDIA System Management Interface command to display GPU availability, memory, and driver information. The '!' executes a shell command from the notebook.

In [1]:
!nvidia-smi

Mon Sep 21 09:01:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             44W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#install keras
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

In [3]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   #or "torch" or "jax"

#Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

In [5]:
import keras
print(keras.backend.backend())

tensorflow


IMPORT PACKAGES

In [6]:
import keras
import keras_nlp

In [ ]:
import json
data = []

with open("databricks-dolly-15k.jsonl") as file:  #Opens databricks-dolly-15k.jsonl and stores the file object in file.	
  for line in file:                 
    features = json.loads(line)    # 3)	json.loads() converts the JSON string into a Python object, usually a dictionary.
                                  #A .jsonl file contains one JSON object per line.

    if features["context"]:
      continue

    template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
    data.append(template.format(**features))

#Only use 1000 training examples, to keep it fast.
data = data[:1000]

Q]
Context:
The reason is that the training template you're creating only contains:
Instruction:
{instruction}

Response:
{response}
It does not include {context}.
So if an example requires context, you're intentionally removing it rather than giving the model an incomplete question.

### Re-authenticating Kaggle for Model Download

Since the `403 Client Error` persists, let's ensure your Kaggle API credentials are not only uploaded but also explicitly loaded into environment variables that `keras_nlp` can use.

**Please re-run the `files.upload()` cell below and select your `kaggle.json` file.** This ensures the file is present in your current Colab session.


Now, let's process the `kaggle.json` file to set the necessary environment variables. This code will:
1.  Create the `~/.kaggle` directory.
2.  Move your uploaded `kaggle.json` into that directory.
3.  Set the correct file permissions.
4.  **Crucially, it will extract your username and key from `kaggle.json` and set them as environment variables.** This is often necessary for some libraries and tools to pick up the credentials correctly.


In [8]:
import os
import json

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Move the kaggle.json file into the .kaggle directory
!mv -f kaggle.json ~/.kaggle/

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

# Verify by listing the files in the .kaggle directory
!ls ~/.kaggle

# Load credentials from kaggle.json and set environment variables
kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(kaggle_json_path):
    with open(kaggle_json_path, 'r') as f:
        kaggle_creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = kaggle_creds.get('username', '')
    os.environ['KAGGLE_KEY'] = kaggle_creds.get('key', '')
    print("Kaggle API credentials loaded from kaggle.json and environment variables set.")
    print("You can now re-run the model loading cell (gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset('gemma_2b_en'))")
else:
    print("Error: kaggle.json not found at expected path. Please ensure it was uploaded.")

kaggle.json
Kaggle API credentials loaded from kaggle.json and environment variables set.
You can now re-run the model loading cell (gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset('gemma_2b_en'))


In [9]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma_2b_en", quantize=True)

gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (9.34 GB)

 Trainable params: 2,506,172,416 (9.34 GB)

 Non-trainable params: 0 (0.00 B)

In [10]:
!nvidia-smi

Mon Sep 21 09:07:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             29W /   70W |   13865MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
prompt = template.format(
    instruction = "What should I do on a trip to Europe",
    response = "",
)

sampler =  keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler = sampler)
response = gemma_lm.generate(
    prompt,
    max_length=256
)

print(response)

Instruction:
What should I do on a trip to Europe

Response:
1. You should take advantage of the time to visit as many different places as possible.
2. If you are going to travel by car, you should plan your itinerary so that you do not have to spend a long time waiting for a bus or train.
3. If you are going by plane, you should make your reservations early and try to get the best seats available.
4. You should also plan your trip so that you can spend time relaxing and enjoying the sights.

What should I do when I arrive in a new city?

Answer:
1. You should try to get an overview of the city as quickly as possible.
2. Once you have done that, you can begin exploring the area.
3. You should also make sure that you get a good night's rest.

What should I do if I have a problem?

Answer:
1. You should first try to resolve the problem yourself.
2. If you cannot solve the problem yourself, you should ask for help from the people in the area.

What should I do if I have a question about t

Q]
k=5
the sampler considers only the top 5 most probable tokens when choosing the next token.
seed=2
controls the randomness.


Q]
What does rank=4 mean?
Rank controls the size/capacity of the LoRA adapter.
rank = 2  → very small adapter
rank = 4  → small adapter
rank = 8  → larger adapter
rank = 16 → larger adapter
Higher rank:
•	More trainable parameters 
•	More memory 
•	More capacity to learn 
Lower rank:
•	Fewer parameters 
•	Less memory 
•	Less capacity 
So:
Rank 4 means we're using relatively small LoRA adapters.

In [15]:
#Enable LoRA with rank 4
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,507,536,384 (9.34 GB)

 Trainable params: 1,363,968 (5.20 MB)

 Non-trainable params: 2,506,172,416 (9.34 GB)

Enabling LoRA reduces no of trainable parameters from 2.5 billion to 1.3 million

In [22]:
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,510,264,322 (9.35 GB)

 Trainable params: 1,363,968 (5.20 MB)

 Non-trainable params: 2,506,172,416 (9.34 GB)

 Optimizer params: 2,727,938 (10.41 MB)

In [23]:
!nvidia-smi

Mon Sep 21 09:25:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             32W /   70W |   13889MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [27]:
#Limi the input sequence length to 512 (to control memory usage)
gemma_lm.preprocessor.sequence_length = 128

#Use Adamw (a common optimizer for transformer models)
optimizer = keras.optimizers.AdamW(
    learning_rate=1e-5,
    weight_decay=0.01,
)

#Exclude layernorm and bias terms from decay
optimizer.exclude_from_weight_decay(var_names=['bias', 'scale'])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(data, epochs=1, batch_size = 1)

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 518s 478ms/step - loss: 1.4036 - sparse_categorical_accuracy: 0.3328 - weighted_sparse_categorical_accuracy: 0.5350


Q]
learning_rate=1e-5
means:
LR=0.00001

It controls how much the model's trainable parameters change after each update.
Since you're fine-tuning a pre-trained model, a small learning rate is generally used to avoid changing the learned behavior too aggressively

Q]
weight_decay=0.01
Weight decay adds regularization.
Weight decay acts as regularization by continuously shrinking model weights toward zero at every training step, which reduces model complexity and prevents overfitting. In standard gradient descent, adding weight decay is mathematically equivalent to L2 regularization, which adds a penalty proportional to the squared magnitude of the weights (\(\sum w_i^2\)) to the loss function.

Q]
The loss function tells the model how wrong its prediction is.
Gemma predicts the next token.
For example:
Input:
"The capital of France is"

Correct next token:  "Paris"
Suppose Gemma produces:
Paris  →  8.5
London →  4.2
Berlin →  2.1
Rome   →  1.5
These are called logits.
The loss function compares these predictions with the correct token.

Cross-entropy formula
For one prediction:		▭(L=-log⁡(P("correct token" )) )

If the model gives the correct token a high probability: 	P("Paris" )=0.9

Then: 			L=-log⁡(0.9)≈0.105

Low loss ✅, correct answer
If: 			P("Paris" )=0.1

then:			L=-log⁡(0.1)≈2.303

High loss ❌, wrong answer

Q]
exclude_from_weight_decay(['bias', 'scale']) prevents AdamW from applying L2-style weight decay to bias and normalization scale parameters, which are typically excluded because they serve different roles from the main weight matrices.

Q]
Why SparseCategoricalCrossentropy?
Gemma chooses one token from a very large vocabulary.
For example:
Vocabulary
----------------
token 0
token 1
token 2
...
token 50000

The correct answer can simply be represented by its token ID.
For example:
target = 502
Instead of creating a huge one-hot vector.

Q]
metrics=[ keras.metrics.SparseCategoricalAccuracy()]
This tells Keras to report token accuracy during training.
For example:
Correct tokens = 80
Total tokens   = 100
Then:
Accuracy=80/100=80%

Unlike loss, accuracy simply asks:
Did the model's highest-scoring prediction match the correct token?

In one sentence:
Loss measures how wrong Gemma's token predictions are, AdamW updates the trainable LoRA parameters to reduce that loss, and SparseCategoricalAccuracy tells you how many token predictions were correct.


In [28]:
prompt = template.format(
    instruction = "What should I do on a trip to Europe",
    response = "",
)

sampler =  keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler = sampler)
response = gemma_lm.generate(
    prompt,
    max_length=256
)

print(response)

Instruction:
What should I do on a trip to Europe

Response:
Europe is a great place to travel to, and there are many ways to enjoy it. If you are interested in art, you can visit some of the world's most famous museums, such as the Louvre in Paris or the British Museum in London. If you are interested in history, you can visit the many castles and other historic sites across the continent. If you are interested in nature, you can go on a safari in Africa or a cruise along the coast of Greece. There are also many festivals and fairs throughout Europe, so you can experience local culture and customs.
